# Clase 6 · Interpretación de modelos en R

Taller práctico de 110 minutos. Selecciona R como entorno de ejecución de Colab. Ejecuta las celdas en orden.

[Clase teórica](https://hizocar.github.io/econometria-usm/clases/clase6/)


## Agenda de 110 minutos

| Tiempo transcurrido | Actividad |
|---|---|
| 000–010 min | Abrir wage1 y reconocer las unidades |
| 010–025 min | De dólares a porcentajes con log(wage) |
| 025–040 min | Nivel-log y log-log sobre una muestra común |
| 040–055 min | Función cuadrática · experiencia y salario |
| 055–070 min | Interacción · educación y experiencia |
| 070–080 min | Repaso visual · elige el modelo |
| 080–090 min | Cambiar unidades y comprobar qué se conserva |
| 090–100 min | Ejercicio en parejas con tenure |
| 100–110 min | Puesta en común y cierre práctico |


## C6-01 · Abrir wage1 y reconocer las unidades

Trabajaremos con wage1, una base del libro de Wooldridge con 526 personas de la encuesta CPS de Estados Unidos de 1976. Cada fila representa una persona. Revisaremos salario, educación y experiencia potencial. El ejercicio final incorpora la antigüedad con el empleador. Ejecuta las celdas en orden con kernel R en Colab.

### Sentencias del ejemplo

**install.packages("wooldridge", repos = "...")**

Instala el paquete desde CRAN. Ejecuta esta línea una vez al preparar el entorno de Colab.

**original <- wooldridge::wage1**

Carga la base wage1 del paquete y la guarda con el nombre original. El símbolo :: permite acceder a la base directamente.

**original[, c("wage", "educ", "exper")]**

Conserva todas las filas y selecciona esas tres columnas. La base original sigue disponible para recuperar tenure.

**head(datos), str(datos) y summary(datos)**

Muestran las primeras filas, la estructura y las estadísticas descriptivas.


### Variables de la base

<table><thead><tr><th>Variable</th><th>Definición y unidad</th></tr></thead><tbody><tr><td><code>wage</code></td><td>Salario promedio por hora, en dólares estadounidenses nominales de 1976.</td></tr><tr><td><code>educ</code></td><td>Años de educación formal.</td></tr><tr><td><code>exper</code></td><td>Años de experiencia laboral potencial, calculados a partir de la edad y la educación.</td></tr><tr><td><code>tenure</code></td><td>Años con el empleador actual. Se usará en el ejercicio en parejas.</td></tr></tbody></table>

### Preparar Colab

Ejecuta esta instalación una vez al preparar el entorno.


In [ ]:
install.packages("wooldridge", repos = "https://cloud.r-project.org/")


In [ ]:
original <- wooldridge::wage1
datos <- original[, c("wage", "educ", "exper")]
head(datos)
str(datos)
summary(datos)


### Resultados del ejemplo

<pre>&#x27;data.frame&#x27;:	526 obs. of  3 variables:
 $ wage : num  3.1 3.24 3 6 5.3 ...
 $ educ : int  11 12 11 8 12 16 18 12 12 17 ...
 $ exper: int  2 22 2 44 7 9 15 5 26 22 ...
      wage             educ           exper      
 Min.   : 0.530   Min.   : 0.00   Min.   : 1.00  
 1st Qu.: 3.330   1st Qu.:12.00   1st Qu.: 5.00  
 Median : 4.650   Median :12.00   Median :13.50  
 Mean   : 5.896   Mean   :12.56   Mean   :17.02  
 3rd Qu.: 6.880   3rd Qu.:14.00   3rd Qu.:26.00  
 Max.   :24.980   Max.   :18.00   Max.   :51.00  </pre>

### Ejercicio

Identifica la unidad de wage y calcula su media y mediana.

### Solución comentada

La media y la mediana se expresan en dólares por hora de 1976. Cada observación corresponde a una persona.


In [ ]:
print(c(media = mean(datos$wage), mediana = median(datos$wage)))


<pre>   media  mediana 
5.896103 4.650000 </pre>


## C6-02 · De dólares a porcentajes con log(wage)

¿Por qué analizar log(wage)? Una diferencia de un dólar por hora representa un cambio proporcional mayor para alguien que gana 3 dólares que para alguien que gana 15. El logaritmo permite estudiar variaciones porcentuales y comprime los valores altos para visualizar la distribución. Compara wage ~ educ con log(wage) ~ educ. En el primero la pendiente mide USD/h por año adicional. En el segundo, 100 por la pendiente aproxima el cambio porcentual por año. El cambio de la curva exponenciada para un salto de d años es 100[exp(b·d) − 1]. Esa curva representa una media geométrica ajustada. La media aritmética del salario requiere una corrección de retransformación.

### Sentencias del ejemplo

**log(wage)**

Calcula el logaritmo natural de cada salario. Solo está definido aquí para valores positivos.

**exp(b_educ) - 1**

exp es la función inversa de log. Convierte un cambio de logaritmos en cambio proporcional de la curva ajustada.

**predict(modelo_log, newdata = nuevos)**

Recibe el modelo y un data frame con la columna educ. Devuelve un vector de predicciones en escala logarítmica, una por fila.

**lines(nuevos$educ, curva, ...)**

Agrega la curva al gráfico abierto. lwd controla el grosor y col el color.


In [ ]:
modelo_nivel <- lm(wage ~ educ, data = datos)
modelo_log <- lm(log(wage) ~ educ, data = datos)
print(coef(modelo_nivel))
print(coef(modelo_log))
b_educ <- coef(modelo_log)["educ"]
print(c(aproximado = 100 * b_educ,
        exacto_curva = 100 * (exp(b_educ) - 1)))
nuevos <- data.frame(educ = 6:18)
ajuste_log <- predict(modelo_log, newdata = nuevos)
curva <- exp(ajuste_log)
plot(wage ~ educ, data = datos, pch = 19, col = "#005F9640",
     xlab = "Educación (años)", ylab = "Salario (USD/h, 1976)")
lines(nuevos$educ, curva, col = "#D6A600", lwd = 3)

# El logaritmo del salario en el eje vertical.
plot(log(wage) ~ educ, data = datos, pch = 19, col = "#005F9640",
     main = "Logaritmo del salario y educación",
     xlab = "Educación (años)", ylab = "Logaritmo del salario (USD/h)")
abline(modelo_log, col = "#D6A600", lwd = 3)


### Resultados del ejemplo

<pre>(Intercept)        educ 
 -0.9048516   0.5413593 
(Intercept)        educ 
 0.58377267  0.08274437 
  aproximado.educ exacto_curva.educ 
         8.274437          8.626409 </pre>

### Ejercicio

Calcula el cambio aproximado y el cambio de la curva exponenciada para dos años adicionales de educación.

### Solución comentada

El porcentaje aproximado se duplica. El cambio porcentual de la curva usa exp(2·b) y puede diferir de dos veces el porcentaje de un año. Usa el control para contrastar saltos de 1 a 4 años.


In [ ]:
print(c(aproximado = 100 * b_educ * 2,
        exacto_curva = 100 * (exp(b_educ * 2) - 1)))


<pre>  aproximado.educ exacto_curva.educ 
         16.54887          17.99697 </pre>


## C6-03 · Nivel-log y log-log sobre una muestra común

¿Por qué usar logaritmos en ambas variables? Así podemos comparar cambios proporcionales. Por ejemplo, pasar de 10 a 11 años de educación es un aumento de 10%, mientras que pasar de 15 a 16 es aproximadamente 6,7%. El modelo log-log resume la asociación entre porcentajes mediante una elasticidad. En log(wage) ~ log(educ), un aumento de 1% en educación se asocia aproximadamente con b% de cambio en la curva salarial. Para contrastar escalas también ajustaremos wage ~ log(educ), donde ese 1% corresponde aproximadamente a b/100 USD/h. Ambos ejemplos usan la misma muestra con educ > 0.

### Sentencias del ejemplo

**datos[datos$educ > 0, ]**

Selecciona educación positiva antes de aplicar log. Informa la diferencia de tamaño muestral.

**log(1.10)**

Es el cambio exacto en log(educ) cuando educ aumenta 10%.

**1.10^b_ll**

Eleva el factor de crecimiento de educación a la elasticidad para obtener el factor de la curva salarial.

**plot(log(wage) ~ log(educ), ...)**

Transforma ambos ejes con el logaritmo natural. La pendiente de la recta en estos ejes es la elasticidad.


In [ ]:
# Primera mirada en las dos escalas logarítmicas.
datos_log <- datos[datos$educ > 0, ]
plot(log(wage) ~ log(educ), data = datos_log, pch = 19, col = "#005F9640",
     main = "Salario y educación en logaritmos",
     xlab = "Logaritmo de educación (años)",
     ylab = "Logaritmo del salario (USD/h)")

print(c(antes = nrow(datos), con_educ_positiva = nrow(datos_log)))
modelo_nivel_log <- lm(wage ~ log(educ), data = datos_log)
modelo_log_log <- lm(log(wage) ~ log(educ), data = datos_log)
print(coef(modelo_nivel_log))
print(coef(modelo_log_log))
b_nl <- coef(modelo_nivel_log)["log(educ)"]
b_ll <- coef(modelo_log_log)["log(educ)"]
print(c(usd_h_por_1pct = b_nl / 100,
        porcentaje_por_1pct = b_ll))
print(c(usd_h_por_10pct = b_nl * log(1.10),
        porcentaje_por_10pct = 100 * (1.10^b_ll - 1)))
abline(modelo_log_log, col = "#D6A600", lwd = 3)


### Resultados del ejemplo

<pre>            antes con_educ_positiva 
              526               524 
(Intercept)   log(educ) 
  -7.460211    5.329674 
(Intercept)   log(educ) 
 -0.4446768   0.8252071 
     usd_h_por_1pct.log(educ) porcentaje_por_1pct.log(educ) 
                   0.05329674                    0.82520715 
     usd_h_por_10pct.log(educ) porcentaje_por_10pct.log(educ) 
                     0.5079722                      8.1826311 </pre>

### Ejercicio

Calcula en ambos modelos los cambios asociados con un aumento de educación de 5%. Indica las unidades.

### Solución comentada

La primera salida está en USD/h y la segunda en porcentaje. Cada interpretación corresponde a una escala de Y distinta. Los R² de modelos con wage y log(wage) resumen variaciones diferentes.


In [ ]:
print(c(usd_h = b_nl * log(1.05),
        porcentaje_curva = 100 * (1.05^b_ll - 1)))


<pre>           usd_h.log(educ) porcentaje_curva.log(educ) 
                 0.2600357                  4.1083494 </pre>


## C6-04 · Función cuadrática · experiencia y salario

Empezamos con una sola variable explicativa, la experiencia potencial, e incorporamos su cuadrado. Queremos representar un salario que aumenta al comienzo y cuya tasa de crecimiento se reduce con la experiencia. En wage = b0 + b1·exper + b2·exper², la pendiente local es b1 + 2·b2·exper. El punto de giro es −b1/(2·b2). Si b2 es negativo, ese punto es un máximo. Revisa si está dentro del rango observado. La derivada describe un cambio local. El salto exacto de e a e+1 agrega además b2.

### Sentencias del ejemplo

**datos$exper2 <- datos$exper^2**

Crea una columna con el cuadrado numérico de la experiencia. Por ejemplo, 10 años se convierten en 100 años cuadrados. R también admite ** como operador de potencia.

**lm(wage ~ exper + exper2, data = datos)**

Estima un intercepto, una pendiente para exper y un coeficiente para su cuadrado. El modelo es lineal en sus coeficientes.

**I(exper^2)**

Es una alternativa válida. I indica que ^ se usa como potencia aritmética dentro de la fórmula. En las fórmulas de modelos, ^ también se usa para expandir interacciones, por ejemplo (x + z)^2. Aquí usamos la columna exper2 para ver cada transformación.

**grilla$exper2 <- grilla$exper^2**

Al predecir, el modelo requiere las dos columnas usadas en la estimación, exper y exper2. Deben corresponder a las mismas experiencias.


In [ ]:
datos$exper2 <- datos$exper^2
modelo_cuad <- lm(wage ~ exper + exper2, data = datos)
b <- coef(modelo_cuad)
print(b)
experiencias <- c(5, 20, 35)
efecto <- b["exper"] + 2 * b["exper2"] * experiencias
print(data.frame(exper = experiencias, efecto_local = efecto))
giro <- -b["exper"] / (2 * b["exper2"])
print(c(giro = giro, minimo = min(datos$exper), maximo = max(datos$exper)))
grilla <- data.frame(exper = seq(min(datos$exper), max(datos$exper), length.out = 100))
grilla$exper2 <- grilla$exper^2
plot(wage ~ exper, data = datos, pch = 19, col = "#005F9640",
     xlab = "Experiencia potencial (años)", ylab = "Salario (USD/h, 1976)")
lines(grilla$exper, predict(modelo_cuad, newdata = grilla),
      col = "#D6A600", lwd = 3)


### Resultados del ejemplo

<pre> (Intercept)        exper       exper2 
 3.725405759  0.298100104 -0.006129887 
  exper efecto_local
1     5   0.23680123
2    20   0.05290461
3    35  -0.13099201
giro.exper     minimo     maximo 
   24.3153     1.0000    51.0000 </pre>

### Ejercicio

Calcula la pendiente local con 10 y 30 años. Compara con el salto predicho entre 10 y 11 años.

### Solución comentada

El coeficiente de exper se interpreta conjuntamente con el cuadrado. La diferencia finita entre 10 y 11 puede diferir de la pendiente local en 10.


In [ ]:
print(b["exper"] + 2 * b["exper2"] * c(10, 30))
p <- predict(modelo_cuad, newdata = data.frame(exper = c(10, 11), exper2 = c(10, 11)^2))
print(p[2] - p[1])


<pre>[1]  0.17550236 -0.06969314
        2 
0.1693725 </pre>


## C6-05 · Interacción · educación y experiencia

Ahora usamos dos variables, educación y experiencia, e incorporamos su producto. La pregunta es si un año adicional de educación se asocia con el mismo cambio salarial para personas con distinta experiencia potencial. En wage = b0 + b1·educ + b2·exper + b3·educ·exper, manteniendo fija la experiencia, el cambio predicho por un año de educación es b1 + b3·exper. Compara las rectas con experiencia fija en 5, 20 y 35 años. Cada recta describe una asociación condicionada al modelo.

### Sentencias del ejemplo

**educ * exper**

En una fórmula de R se expande a educ + exper + educ:exper. El término con : es el producto de ambas variables.

**comparar$predicho <- predict(...)**

Agrega a cada fila la predicción correspondiente a su educación y experiencia.

**match(e, experiencias)**

Entrega la posición de e en el vector para escoger el color de esa recta.

**legend(..., bty = "n")**

Identifica los colores de las tres rectas. bty = "n" dibuja la leyenda sin marco.


In [ ]:
modelo_int <- lm(wage ~ educ * exper, data = datos)
bi <- coef(modelo_int)
print(bi)
experiencias <- c(5, 20, 35)
efecto_educ <- bi["educ"] + bi["educ:exper"] * experiencias
print(data.frame(exper = experiencias, usd_h_por_ano_educ = efecto_educ))
comparar <- data.frame(educ = c(12, 13, 12, 13), exper = c(5, 5, 20, 20))
comparar$predicho <- predict(modelo_int, newdata = comparar)
print(comparar)
plot(wage ~ educ, data = datos, pch = 19, col = "#005F9640",
     xlab = "Educación (años)", ylab = "Salario (USD/h, 1976)")
for (e in experiencias) {
  nuevos <- data.frame(educ = 6:18, exper = e)
  lines(nuevos$educ, predict(modelo_int, newdata = nuevos),
        col = c("#005F96", "#D6A600", "#B2292E")[match(e, experiencias)], lwd = 2)
}
legend("topleft", legend = paste(experiencias, "años de experiencia"),
       col = c("#005F96", "#D6A600", "#B2292E"), lwd = 2, bty = "n")


### Resultados del ejemplo

<pre> (Intercept)         educ        exper   educ:exper 
-2.859915627  0.601735470  0.045768911  0.002062345 
  exper usd_h_por_ano_educ
1     5          0.6120472
2    20          0.6429824
3    35          0.6739175
  educ exper predicho
1   12     5 4.713495
2   13     5 5.325542
3   12    20 5.771251
4   13    20 6.414233</pre>

### Ejercicio

Calcula la diferencia predicha al pasar de 12 a 13 años de educación con experiencia fija en 35. Contrástala con la fórmula del efecto.

### Solución comentada

Los dos cálculos coinciden. b_educ es la pendiente cuando exper = 0. Para otra experiencia hay que sumar el término de interacción.


In [ ]:
p <- predict(modelo_int, newdata = data.frame(educ = c(12, 13), exper = c(35, 35)))
print(c(diferencia = p[2] - p[1],
        formula = bi["educ"] + bi["educ:exper"] * 35))


<pre>diferencia.2 formula.educ 
   0.6739175    0.6739175 </pre>


## C6-06 · Repaso visual · elige el modelo

Elige una de las cuatro cajas. Observa cómo se dibuja la curva y relaciona la escala de los ejes con la interpretación. Primero anticipa las unidades de la pendiente y después revisa la explicación. Las curvas usan los mismos modelos estimados en R en las láminas 02 a 05.

### Sentencias del ejemplo

**coef(modelo)**

Extrae las pendientes de los modelos ya estimados. La escala del modelo determina cómo se interpreta cada salida.


In [ ]:
print(c(porcentaje_por_ano_educ = 100 * coef(modelo_log)["educ"],
        elasticidad = coef(modelo_log_log)["log(educ)"],
        pendiente_exper_10 = b["exper"] + 2 * b["exper2"] * 10,
        pendiente_educ_exper_20 = bi["educ"] + bi["educ:exper"] * 20))


### Resultados del ejemplo

<pre>porcentaje_por_ano_educ.educ        elasticidad.log(educ) 
                   8.2744366                    0.8252071 
    pendiente_exper_10.exper pendiente_educ_exper_20.educ 
                   0.1755024                    0.6429824 </pre>

### Ejercicio

Sin ejecutar, explica las unidades de cada fila. ¿Qué efecto cambia con la experiencia y qué modelo resume una elasticidad?

### Solución comentada

Log-nivel entrega un porcentaje aproximado por año. Log-log entrega una elasticidad. En la cuadrática la pendiente cambia con la propia experiencia. En la interacción la pendiente de educación cambia con la experiencia que se mantiene fija.


In [ ]:
print(data.frame(
  modelo = c("Log-nivel", "Log-log", "Cuadrático en exper", "Interacción"),
  unidad = c("% aproximado por año de educación", "% aproximado por 1% de educación",
             "USD/h por año de experiencia, en 10 años",
             "USD/h por año de educación, con 20 años de experiencia")))


<pre>               modelo                                                 unidad
1           Log-nivel                      % aproximado por año de educación
2             Log-log                       % aproximado por 1% de educación
3 Cuadrático en exper               USD/h por año de experiencia, en 10 años
4         Interacción USD/h por año de educación, con 20 años de experiencia</pre>


## C6-07 · Cambiar unidades y comprobar qué se conserva

Usa un tipo de cambio hipotético de 1.000 pesos por dólar para practicar reescalamiento. Al multiplicar Y por 1.000, el intercepto y la pendiente se multiplican por 1.000, y los residuos también. La suma de cuadrados residuales se multiplica por un millón. Si educación pasa de años a meses, su pendiente se divide por 12. En los tres casos, el R² coincide.

### Sentencias del ejemplo

**rbind(...)**

Apila los vectores de coeficientes como filas para compararlos. Cada fila conserva el orden intercepto y pendiente.

**summary(modelo)$r.squared**

Extrae el R² del resumen de cada modelo.

**resid(modelo) y fitted(modelo)**

Devuelven, respectivamente, los residuos y los valores ajustados para las personas usadas en el modelo.

**all.equal(a, b)**

Compara resultados numéricos admitiendo pequeñas diferencias de redondeo. TRUE indica coincidencia dentro de la tolerancia.


In [ ]:
datos$salario_pesos <- 1000 * datos$wage
datos$educ_meses <- 12 * datos$educ
modelo_pesos <- lm(salario_pesos ~ educ, data = datos)
modelo_meses <- lm(wage ~ educ_meses, data = datos)
print(rbind(dolares = coef(modelo_nivel),
            pesos = coef(modelo_pesos), meses = coef(modelo_meses)))
print(c(R2_dolares = summary(modelo_nivel)$r.squared,
        R2_pesos = summary(modelo_pesos)$r.squared,
        R2_meses = summary(modelo_meses)$r.squared))
print(c(SCR_dolares = sum(resid(modelo_nivel)^2),
        SCR_pesos = sum(resid(modelo_pesos)^2)))
print(all.equal(fitted(modelo_pesos) / 1000, fitted(modelo_nivel)))
print(all.equal(fitted(modelo_meses), fitted(modelo_nivel)))


### Resultados del ejemplo

<pre>         (Intercept)         educ
dolares   -0.9048516   0.54135925
pesos   -904.8516120 541.35925467
meses     -0.9048516   0.04511327
R2_dolares   R2_pesos   R2_meses 
 0.1647575  0.1647575  0.1647575 
 SCR_dolares    SCR_pesos 
5.980682e+03 5.980682e+09 
[1] TRUE
[1] TRUE</pre>

### Ejercicio

Expresa wage en centavos de dólar y vuelve a estimar el modelo. ¿Por cuánto se multiplican la pendiente y la SCR?

### Solución comentada

Los coeficientes se multiplican por 100 y la SCR por 10.000. El R² conserva su valor. El selector permite comparar unidades.


In [ ]:
datos$centavos <- 100 * datos$wage
modelo_centavos <- lm(centavos ~ educ, data = datos)
print(coef(modelo_centavos))
print(sum(resid(modelo_centavos)^2) / sum(resid(modelo_nivel)^2))


<pre>(Intercept)        educ 
  -90.48516    54.13593 
[1] 10000</pre>


## C6-08 · Ejercicio en parejas con tenure

Cada pareja abre la base wage1 y escribe su código. tenure son años con el empleador actual. El desafío cambia el regresor para practicar la interpretación sin reutilizar de memoria el coeficiente de educ. Usa 5 minutos para programar y 5 para comparar interpretaciones entre parejas.

### Sentencias del ejemplo

**original[, c("wage", "tenure")]**

Recupera estas dos columnas de la base completa. datos contiene inicialmente solo wage, educ y exper.


In [ ]:
# Punto de partida. Ejecuta la carga de la lámina 01 primero.
practica <- original[, c("wage", "tenure")]
practica <- practica[complete.cases(practica) & practica$wage > 0, ]
head(practica)
summary(practica)


### Resultados del ejemplo

<pre>      wage            tenure      
 Min.   : 0.530   Min.   : 0.000  
 1st Qu.: 3.330   1st Qu.: 0.000  
 Median : 4.650   Median : 2.000  
 Mean   : 5.896   Mean   : 5.105  
 3rd Qu.: 6.880   3rd Qu.: 7.000  
 Max.   :24.980   Max.   :44.000  </pre>

### Ejercicio

Ajusta log(wage) ~ tenure. Calcula el cambio porcentual aproximado y el de la curva para tres años adicionales. Dibuja la nube de tenure y log(wage), con la recta superpuesta. Finalmente expresa tenure en meses y anticipa cómo cambia la pendiente.

### Solución comentada

La pendiente por mes es la pendiente por año dividida por 12. Los cambios porcentuales para el mismo período de tiempo coinciden. El gráfico tiene log(wage) en el eje vertical.


In [ ]:
modelo_tenure <- lm(log(wage) ~ tenure, data = practica)
bt <- coef(modelo_tenure)["tenure"]
print(coef(modelo_tenure))
print(c(aproximado = 100 * bt * 3,
        exacto_curva = 100 * (exp(bt * 3) - 1)))
plot(log(wage) ~ tenure, data = practica, pch = 19, col = "#005F9640",
     xlab = "Antigüedad con el empleador (años)", ylab = "Logaritmo del salario")
abline(modelo_tenure, col = "#D6A600", lwd = 3)
practica$meses <- 12 * practica$tenure
modelo_tenure_meses <- lm(log(wage) ~ meses, data = practica)
print(coef(modelo_tenure_meses))


<pre>(Intercept)      tenure 
 1.50100711  0.02395138 
  aproximado.tenure exacto_curva.tenure 
           7.185414            7.449861 
(Intercept)       meses 
1.501007108 0.001995948 </pre>


## C6-09 · Puesta en común y cierre práctico

Cada pareja explica una salida de su ejercicio durante 8 minutos. En los 7 restantes, verifica los cuatro cambios de interpretación con este código. Primero completa la frase de cada fila y después abre la solución. La clase suma 110 minutos de trabajo y revisión.

### Sentencias del ejemplo

**coef(modelo)["nombre"]**

Extrae una pendiente mediante su nombre exacto. Cada modelo define su propia escala para Y y X.


In [ ]:
# Recupera las cuatro pendientes vistas hoy.
print(c(nivel_nivel = coef(modelo_nivel)["educ"],
        log_nivel = coef(modelo_log)["educ"],
        nivel_log = coef(modelo_nivel_log)["log(educ)"],
        log_log = coef(modelo_log_log)["log(educ)"]))


### Resultados del ejemplo

<pre>   nivel_nivel.educ      log_nivel.educ nivel_log.log(educ)   log_log.log(educ) 
         0.54135925          0.08274437          5.32967449          0.82520715 </pre>

### Ejercicio

Completa las unidades. Nivel-nivel, por un año adicional de educación. Log-nivel, por un año adicional. Nivel-log, por 1% más de educación. Log-log, por 1% más de educación. ¿En cuál de las cuatro frases la pendiente ya es una elasticidad?

### Solución comentada

Nivel-nivel entrega USD/h por año. Log-nivel entrega aproximadamente porcentaje por año. Nivel-log entrega aproximadamente USD/h por 1%. Log-log entrega aproximadamente porcentaje por 1% y su pendiente es la elasticidad.


In [ ]:
print(c(usd_h_por_ano = coef(modelo_nivel)["educ"],
        porcentaje_por_ano = 100 * coef(modelo_log)["educ"],
        usd_h_por_1pct = coef(modelo_nivel_log)["log(educ)"] / 100,
        porcentaje_por_1pct = coef(modelo_log_log)["log(educ)"]))


<pre>           usd_h_por_ano.educ       porcentaje_por_ano.educ 
                   0.54135925                    8.27443659 
     usd_h_por_1pct.log(educ) porcentaje_por_1pct.log(educ) 
                   0.05329674                    0.82520715 </pre>
